# Length Generalization Experiments
**2 tasks × 4 PE schemes × 3 seeds = 24 runs**

Set runtime to **GPU** (Runtime → Change runtime type → T4) for ~10 min total.

CPU works too (~45 min).

Run all cells top to bottom. The last cell downloads the ZIP with all results, figures, and CSVs.

In [ ]:
# Cell 1: Setup
import math, json, os, time, itertools
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats as sp_stats
from pathlib import Path

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

BASE = Path('lengen_results')
RESULTS = BASE / 'results'
FIGURES = BASE / 'figures'
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

In [ ]:
# Cell 2: Vocabulary & Data
DIGITS = list('0123456789')
VOCAB = DIGITS + ['+', '=', 'P', 'E']
STOI = {c: i for i, c in enumerate(VOCAB)}
V = len(VOCAB)
PAD = STOI['P']

def make_task(task, n, rng):
    s = [DIGITS[rng.integers(0, 10)] for _ in range(n)]
    prompt = s + ['=']
    if task == 'copy':
        target = s + ['E']
    elif task == 'reverse':
        target = s[::-1] + ['E']
    else:
        raise ValueError(task)
    return prompt, target

def encode_batch(items, Lmax):
    B = len(items)
    x = np.full((B, Lmax), PAD, dtype=np.int64)
    tgt = np.full((B, Lmax), PAD, dtype=np.int64)
    mask = np.zeros((B, Lmax), dtype=np.float32)
    for i, (p, t) in enumerate(items):
        seq = p + t
        ids = [STOI[c] for c in seq]
        n = len(ids)
        x[i, :n] = ids
        for j in range(n - 1):
            tgt[i, j] = ids[j + 1]
            if j >= len(p) - 1:
                mask[i, j] = 1.0
    return (torch.from_numpy(x).to(DEVICE),
            torch.from_numpy(tgt).to(DEVICE),
            torch.from_numpy(mask).to(DEVICE))

print('Data functions ready.')

In [ ]:
# Cell 3: Model
class Head(nn.Module):
    def __init__(self, d, dh, pe, Lmax):
        super().__init__()
        self.q = nn.Linear(d, dh, bias=False)
        self.k = nn.Linear(d, dh, bias=False)
        self.v = nn.Linear(d, dh, bias=False)
        self.dh, self.pe = dh, pe
        self.Lmax = Lmax

    def _rope(self, x):
        B, L, dh = x.shape
        half = dh // 2
        pos = torch.arange(L, device=x.device).float()
        inv = 1.0 / (10000 ** (torch.arange(0, half, device=x.device).float() / half))
        ang = pos[:, None] * inv[None, :]
        cos, sin = torch.cos(ang), torch.sin(ang)
        x1, x2 = x[..., :half], x[..., half:]
        return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)

    def forward(self, x, alibi_slope=None, need_attn=False):
        B, L, _ = x.shape
        q, k, v = self.q(x), self.k(x), self.v(x)
        if self.pe == 'rope':
            q, k = self._rope(q), self._rope(k)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.dh)
        if self.pe == 'alibi':
            dist = torch.arange(L, device=x.device)[None, :] - torch.arange(L, device=x.device)[:, None]
            att = att + alibi_slope * (-torch.abs(dist).float())
        causal = torch.triu(torch.ones(L, L, device=x.device), 1).bool()
        att = att.masked_fill(causal[None], float('-inf'))
        a = F.softmax(att, dim=-1)
        out = a @ v
        return (out, a) if need_attn else (out, None)

class Block(nn.Module):
    def __init__(self, d, nh, pe, Lmax, ff):
        super().__init__()
        self.heads = nn.ModuleList([Head(d, d // nh, pe, Lmax) for _ in range(nh)])
        self.proj = nn.Linear(d, d)
        self.ln1 = nn.LayerNorm(d); self.ln2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Linear(ff, d))
        self.register_buffer('slopes', torch.tensor(
            [2 ** (-8 * (i + 1) / nh) for i in range(nh)]))
        self.pe = pe

    def forward(self, x, need_attn=False):
        h = self.ln1(x); outs = []; attns = []
        for i, head in enumerate(self.heads):
            o, a = head(h, alibi_slope=self.slopes[i], need_attn=need_attn)
            outs.append(o); attns.append(a)
        x = x + self.proj(torch.cat(outs, dim=-1))
        x = x + self.mlp(self.ln2(x))
        return x, attns

class TinyGPT(nn.Module):
    def __init__(self, d=64, nh=4, nl=2, pe='nope', Lmax=48, ff=128):
        super().__init__()
        self.tok = nn.Embedding(V, d)
        self.pe_name = pe
        if pe == 'ape':
            self.pos = nn.Embedding(Lmax, d)
        self.blocks = nn.ModuleList([Block(d, nh, pe, Lmax, ff) for _ in range(nl)])
        self.lnf = nn.LayerNorm(d)
        self.head = nn.Linear(d, V)
        self.Lmax = Lmax

    def forward(self, x, need_attn=False):
        h = self.tok(x)
        if self.pe_name == 'ape':
            L = x.shape[1]
            h = h + self.pos(torch.arange(L, device=x.device))[None]
        attn_all = []
        for b in self.blocks:
            h, a = b(h, need_attn=need_attn)
            attn_all.append(a)
        return self.head(self.lnf(h)), attn_all

print('Model classes ready.')

In [ ]:
# Cell 4: Translation Invariance Score
@torch.no_grad()
def translation_invariance(model, items, Lmax):
    """R² of offset-only model of attention: 1.0 = fully relative, low = absolute."""
    x, _, _ = encode_batch(items, Lmax)
    L0 = max(len(p) + len(t) for p, t in items)
    model.eval()
    _, attn_all = model(x, need_attn=True)
    scores = []
    for layer in attn_all:
        for A in layer:
            Abar = A[:, :L0, :L0].mean(0).cpu().numpy()
            ii, jj = np.tril_indices(L0, 0)
            vals = Abar[ii, jj]
            off = ii - jj
            m = np.zeros_like(vals)
            for d_val in np.unique(off):
                m[off == d_val] = vals[off == d_val].mean()
            ss_res = ((vals - m) ** 2).sum()
            ss_tot = ((vals - vals.mean()) ** 2).sum() + 1e-12
            scores.append(1.0 - ss_res / ss_tot)
    model.train()
    return float(np.mean(scores))

print('Invariance metric ready.')

In [ ]:
# Cell 5: Training & Evaluation Functions
def sample_batch(B, lo, hi, rng, Lmax, task='copy'):
    items = []
    for _ in range(B):
        n = int(rng.integers(lo, hi + 1))
        items.append(make_task(task, n, rng))
    return encode_batch(items, Lmax)

@torch.no_grad()
def eval_len(model, n, rng, Lmax, n_eval=400, task='copy'):
    model.eval()
    correct = 0
    items = [make_task(task, n, rng) for _ in range(n_eval)]
    x, tgt, mask = encode_batch(items, Lmax)
    logits, _ = model(x)
    pred = logits.argmax(-1)
    m = mask.bool()
    for i in range(x.shape[0]):
        idx = m[i].nonzero().squeeze(-1)
        if len(idx) == 0:
            continue
        if torch.equal(pred[i, idx], tgt[i, idx]):
            correct += 1
    model.train()
    return correct / x.shape[0]

def run_experiment(pe='nope', seed=0, train_lo=1, train_hi=10, test_hi=18,
                   d=64, nh=4, nl=2, Lmax=48, ff=128, steps=3000, B=128,
                   lr=5e-4, task='copy', tag='main'):
    torch.manual_seed(seed); np.random.seed(seed)
    rng = np.random.default_rng(seed)
    model = TinyGPT(d=d, nh=nh, nl=nl, pe=pe, Lmax=Lmax, ff=ff).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, s / 200) * (0.5 ** (s / max(1, steps))))
    t0 = time.time()
    for s in range(steps):
        x, tgt, mask = sample_batch(B, train_lo, train_hi, rng, Lmax, task=task)
        logits, _ = model(x)
        loss = (F.cross_entropy(logits.reshape(-1, V), tgt.reshape(-1),
                                reduction='none').reshape(tgt.shape) * mask).sum() / mask.sum()
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
    # per-length accuracy
    accs = {}
    for n in range(1, test_hi + 1):
        accs[n] = eval_len(model, n, np.random.default_rng(9000 + n), Lmax, task=task)
    # translation invariance
    prng = np.random.default_rng(5000 + seed)
    probe_items = [make_task(task, train_hi, prng) for _ in range(64)]
    inv = translation_invariance(model, probe_items, Lmax)
    out = dict(pe=pe, seed=seed, task=task, train_lo=train_lo, train_hi=train_hi,
               test_hi=test_hi, steps=steps, d=d, nl=nl, nh=nh,
               accs={str(k): v for k, v in accs.items()}, invariance=inv,
               train_time=time.time() - t0, final_loss=float(loss.detach()))
    fn = RESULTS / f'lengen_{task}_{pe}_{tag}_s{seed}.json'
    with open(fn, 'w') as f:
        json.dump(out, f, indent=2)
    tr = np.mean([accs[n] for n in range(train_lo, train_hi + 1)])
    ex = np.mean([accs[n] for n in range(train_hi + 1, test_hi + 1)])
    print(f'[{task[:3]} {pe:5s} s{seed}] train={tr:.3f} extrap={ex:.3f} '
          f'inv={inv:.3f} ({out["train_time"]:.0f}s)', flush=True)
    return model, out

print('Training functions ready.')

In [ ]:
# Cell 6: RUN ALL 24 EXPERIMENTS
# This is the main cell — takes ~10 min on T4 GPU, ~45 min on CPU

TASKS = ['copy', 'reverse']
PES = ['ape', 'nope', 'alibi', 'rope']
SEEDS = [0, 1, 2]

total = len(TASKS) * len(PES) * len(SEEDS)
done = 0
t_start = time.time()

for task, pe, seed in itertools.product(TASKS, PES, SEEDS):
    fn = RESULTS / f'lengen_{task}_{pe}_main_s{seed}.json'
    if fn.exists():
        print(f'[SKIP] {fn.name} already exists')
        done += 1
        continue
    done += 1
    print(f'\n--- [{done}/{total}] {task} × {pe} × seed={seed} ---')
    run_experiment(pe=pe, seed=seed, task=task, tag='main')

elapsed = time.time() - t_start
print(f'\n{"=" * 60}')
print(f'All {total} experiments done in {elapsed:.0f}s ({elapsed/60:.1f} min)')
print(f'{"=" * 60}')

In [ ]:
# Cell 7: Generate Fig 1 — Per-length accuracy curves

def load_results():
    records = []
    for fn in sorted(RESULTS.glob('lengen_*_main_*.json')):
        with open(fn) as f:
            records.append(json.load(f))
    return records

records = load_results()
print(f'Loaded {len(records)} result files')

PE_LABELS = {'ape': 'APE (Learned Absolute)', 'nope': 'NoPE (No Encoding)',
             'alibi': 'ALiBi (Linear Bias)', 'rope': 'RoPE (Rotary)'}
PE_COLORS = {'ape': '#d62728', 'nope': '#7f7f7f', 'alibi': '#2ca02c', 'rope': '#1f77b4'}
PE_MARKERS = {'ape': 's', 'nope': 'D', 'alibi': 'o', 'rope': '^'}

test_hi = max(int(k) for r in records for k in r['accs'].keys())
train_hi = records[0]['train_hi']

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ti, task in enumerate(['copy', 'reverse']):
    ax = axes[ti]
    ax.axvspan(0.5, train_hi + 0.5, color='#f0f0f0', zorder=0)
    ax.axvline(train_hi + 0.5, color='gray', ls='--', lw=0.8, label='Train boundary')
    for pe in PES:
        runs = [r for r in records if r['task'] == task and r['pe'] == pe]
        if not runs:
            continue
        lengths = sorted(int(k) for k in runs[0]['accs'].keys())
        mat = np.array([[r['accs'][str(n)] for n in lengths] for r in runs])
        mu = mat.mean(0)
        std = mat.std(0)
        ax.plot(lengths, mu, color=PE_COLORS[pe], marker=PE_MARKERS[pe],
                markersize=5, label=PE_LABELS[pe], linewidth=1.8)
        ax.fill_between(lengths, mu - std, mu + std, color=PE_COLORS[pe], alpha=0.15)
    ax.set_title(f'Task: {task.capitalize()}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Sequence Length', fontsize=11)
    ax.set_xlim(0.5, test_hi + 0.5)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xticks(range(2, test_hi + 1, 2))
    if ti == 0:
        ax.set_ylabel('Exact-Match Accuracy', fontsize=11)
    ax.legend(fontsize=8, loc='center right')
    ax.grid(True, alpha=0.3)
plt.suptitle('Length Generalization: Per-Length Exact-Match Accuracy',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'fig1_accuracy_curves.pdf', bbox_inches='tight')
plt.savefig(FIGURES / 'fig1_accuracy_curves.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved fig1_accuracy_curves.pdf/png')

In [ ]:
# Cell 8: Generate Fig 2 — Invariance vs Extrapolation scatter

fig, ax = plt.subplots(figsize=(6, 5))
inv_all, ext_all = [], []
for r in records:
    accs = r['accs']
    ext = np.mean([accs[str(n)] for n in range(train_hi + 1, test_hi + 1)])
    inv = r['invariance']
    pe = r['pe']
    ax.scatter(inv, ext, color=PE_COLORS[pe], marker=PE_MARKERS[pe],
               s=80, edgecolors='black', linewidths=0.5, zorder=3,
               label=PE_LABELS[pe])
    inv_all.append(inv); ext_all.append(ext)

inv_arr, ext_arr = np.array(inv_all), np.array(ext_all)
r_val, p_val = sp_stats.pearsonr(inv_arr, ext_arr)
slope, intercept = np.polyfit(inv_arr, ext_arr, 1)
x_fit = np.linspace(inv_arr.min() - 0.02, inv_arr.max() + 0.02, 100)
ax.plot(x_fit, slope * x_fit + intercept, 'k--', lw=1, alpha=0.6)
ax.text(0.05, 0.95, f'r = {r_val:.3f}  (p = {p_val:.2e})',
        transform=ax.transAxes, fontsize=11, va='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), fontsize=9)
ax.set_xlabel('Translation-Invariance Score (Offset R²)', fontsize=11)
ax.set_ylabel('Mean Extrapolation Accuracy', fontsize=11)
ax.set_title('Attention Invariance Predicts Length Generalization',
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES / 'fig2_invariance_scatter.pdf', bbox_inches='tight')
plt.savefig(FIGURES / 'fig2_invariance_scatter.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved fig2  |  r = {r_val:.3f}, p = {p_val:.2e}')

In [ ]:
# Cell 9: Generate Fig 3 — Attention heatmaps (APE vs ALiBi)
# Trains 2 small models fresh (fast)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for col, pe in enumerate(['ape', 'alibi']):
    model, _ = run_experiment(pe=pe, seed=42, steps=2000, train_hi=8,
                              test_hi=14, task='copy', tag='attn')
    model.eval()
    for row, n in enumerate([8, 13]):
        items = [make_task('copy', n, np.random.default_rng(7777 + n))
                 for _ in range(64)]
        x, _, _ = encode_batch(items, model.Lmax)
        _, attn_all = model(x, need_attn=True)
        A = torch.stack(attn_all[0]).mean(0).mean(0).cpu().numpy()
        L = 2 * n + 2
        ax = axes[row, col]
        im = ax.imshow(A[:L, :L], cmap='Blues', vmin=0, vmax=0.5, aspect='auto')
        label = 'Train (len=8)' if n == 8 else 'Extrap (len=13)'
        ax.set_title(f'{pe.upper()} — {label}', fontsize=11)
        ax.set_xlabel('Key position')
        ax.set_ylabel('Query position')
plt.suptitle('Attention Patterns: APE vs ALiBi at Train and Extrapolation Lengths',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'fig3_attention.pdf', bbox_inches='tight')
plt.savefig(FIGURES / 'fig3_attention.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved fig3_attention.pdf/png')

In [ ]:
# Cell 10: Summary CSV
import csv

records = load_results()
with open(BASE / 'summary.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['task', 'pe', 'seed', 'train_acc', 'extrap_acc', 'invariance',
                'train_time_s'] + [f'len_{n}' for n in range(1, test_hi + 1)])
    for r in sorted(records, key=lambda x: (x['task'], x['pe'], x['seed'])):
        accs = r['accs']
        tr = np.mean([accs[str(n)] for n in range(r['train_lo'], train_hi + 1)])
        ex = np.mean([accs[str(n)] for n in range(train_hi + 1, test_hi + 1)])
        row = [r['task'], r['pe'], r['seed'], f'{tr:.4f}', f'{ex:.4f}',
               f'{r["invariance"]:.4f}', f'{r["train_time"]:.1f}']
        row += [f'{accs[str(n)]:.4f}' for n in range(1, test_hi + 1)]
        w.writerow(row)
print(f'Saved summary.csv with {len(records)} rows')

# Print summary table
print(f'\n{"Task":<8} {"PE":<6} {"Train Acc":>10} {"Extrap Acc":>11} {"Invariance":>11}')
print('-' * 50)
for task in ['copy', 'reverse']:
    for pe in PES:
        runs = [r for r in records if r['task'] == task and r['pe'] == pe]
        if not runs:
            continue
        tr_vals = [np.mean([r['accs'][str(n)] for n in range(1, train_hi+1)]) for r in runs]
        ex_vals = [np.mean([r['accs'][str(n)] for n in range(train_hi+1, test_hi+1)]) for r in runs]
        iv_vals = [r['invariance'] for r in runs]
        print(f'{task:<8} {pe:<6} {np.mean(tr_vals):>9.3f}±{np.std(tr_vals):.3f}'
              f'{np.mean(ex_vals):>10.3f}±{np.std(ex_vals):.3f}'
              f'{np.mean(iv_vals):>10.3f}±{np.std(iv_vals):.3f}')

In [ ]:
# Cell 11: Create ZIP and download
import shutil

zip_path = shutil.make_archive('lengen_results', 'zip', '.', 'lengen_results')
print(f'ZIP created: {zip_path}')
print(f'Contents:')
for p in sorted(BASE.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(BASE)}  ({p.stat().st_size / 1024:.1f} KB)')

# Download
try:
    from google.colab import files
    files.download('lengen_results.zip')
    print('\n✅ Download started!')
except ImportError:
    print('\n(Not in Colab — find the ZIP at:', zip_path, ')')